# CRISPRa sgRNA re-annotation — Phase 1-2: unified library, gene-ID reconciliation, frameshift-twin dedup

Genome-wide CRISPRa library (hCRISPRa-v2 / Calabrese Set A + Set B). Two IDT pools of ~2 guides/gene
combine to **4 guides/gene**. This notebook covers the library-intrinsic steps that need no genome
alignment (base conda only):

- **Phase 1** — build one unified per-guide table from `CRISPRa_pool1.csv` + `CRISPRa_pool2.csv`,
  separating the `NO-TARGET` non-targeting block from the targeting guides.
- **Phase 1.5** — reconcile the library's legacy gene **symbols** (no Ensembl IDs, ~7% outdated) to
  current `gene_id` + approved symbol via GENCODE v48 + HGNC previous/alias symbols. This must precede
  twin detection: a real same-gene twin can carry an old label on one guide and the new label on the
  other, which would otherwise look cross-gene.
- **Phase 2** — detect **frameshift twins** keyed on the reconciled `gene_id`: pairs of guides targeting
  the *same gene* whose 20 bp protospacers share a 19 bp window at a **1 bp offset**. These are redundant
  guides (same genomic site, tiled by 1 bp) that an exact-`last19bp` dedup misses, and whose RHS capture
  probes cross-hybridize. Collapse each twin group to one representative and flag the rest.

Alignment-based validation (that twins map to adjacent loci), locus-authoritative gene assignment for the
flagged ambiguous/unresolved symbols, and CRISPRa on-target scoring are **Phases 3-6** (separate notebook,
needs the `crispra_sgrna` env). Helpers live in `sgRNAalign_util.py`. Outputs go to `results/`.

In [1]:
import os
import pandas as pd
import numpy as np

from sgRNAalign_util import (
    reconcile_symbols_to_geneid, frameshift_twin_pairs, collapse_twin_groups,
)

# Paths are relative to this notebook (3_codes/PerturbSeq_Analysis_pipeline/src/5_sgRNA_annotation/)
LIB = '../../../../1_files/CRISPRa_library_construction'
GENOME = 'genome'          # local: gene_annotations_hg38.parquet (GENCODE v48), hgnc_complete_set.txt
RESULTS = 'results'
os.makedirs(RESULTS, exist_ok=True)

In [2]:
# --- Acquire external reference inputs (idempotent: download/copy only if missing) ---
import shutil
import urllib.request

REF_GENOME = ('/Users/rzhu/Gladstone Dropbox/Ronghui Zhu/GRNPerturbSeq/4_codes/'
              'GWT_perturbseq_analysis/src/5_sgRNA_annotation/genome')

# HGNC complete set (previous/alias symbol -> current symbol + Ensembl id) for name reconciliation
hgnc_path = f'{GENOME}/hgnc_complete_set.txt'
if not os.path.exists(hgnc_path):
    print('downloading HGNC complete set (~16 MB)...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt',
        hgnc_path)

# GENCODE v48 primary gene annotation (gene_name -> gene_id); reuse the sibling project's parsed parquet
gene_ann = f'{GENOME}/gene_annotations_hg38.parquet'
if not os.path.exists(gene_ann):
    print('copying gene_annotations_hg38.parquet from reference build...')
    shutil.copy(f'{REF_GENOME}/gene_annotations_hg38.parquet', gene_ann)

print('inputs ready:', {'hgnc': os.path.exists(hgnc_path), 'gene_annotations': os.path.exists(gene_ann)})

downloading HGNC complete set (~16 MB)...


copying gene_annotations_hg38.parquet from reference build...
inputs ready: {'hgnc': True, 'gene_annotations': True}


## Phase 1 — Build the unified CRISPRa sgRNA table

Pool 1 = Set A (guide ranks -1, -2); Pool 2 = Set B (ranks -3, -4). `Barcode Sequence` is the 20 bp
protospacer; `last19bp` = `protospacer[1:]` (the 5' base is an appended/unreliable base in hCRISPRa-v2,
so 19 bp is the discriminating unit). We concatenate both pools and split off the `NO-TARGET`
non-targeting block.

In [3]:
def load_pool(path, pool_name):
    df = pd.read_csv(path)
    df = df.rename(columns={
        'Barcode Sequence': 'protospacer',
        'Annotated Gene Symbol': 'target_gene_symbol',
    })
    df['pool'] = pool_name
    df['protospacer'] = df['protospacer'].str.upper()
    df['last19bp'] = df['last19bp'].str.upper()
    return df[['guide_id', 'protospacer', 'last19bp', 'target_gene_symbol',
               'Set', 'guide_rank', 'gc_fraction', 'pool']]

sg = pd.concat([
    load_pool(f'{LIB}/CRISPRa_pool1.csv', 'pool1'),
    load_pool(f'{LIB}/CRISPRa_pool2.csv', 'pool2'),
], ignore_index=True)

# Integrity checks
assert (sg['last19bp'] == sg['protospacer'].str[1:]).all(), 'last19bp != protospacer[1:]'
assert sg['protospacer'].str.len().eq(20).all(), 'not all protospacers are 20 bp'
assert not sg['guide_id'].duplicated().any(), 'duplicate guide_id across pools'

targeting = sg[sg['target_gene_symbol'] != 'NO-TARGET'].reset_index(drop=True)
ntc = sg[sg['target_gene_symbol'] == 'NO-TARGET'].reset_index(drop=True)

print(f'combined guides     : {len(sg):,}')
print(f'targeting guides    : {len(targeting):,}  across {targeting.target_gene_symbol.nunique():,} genes')
print(f'NO-TARGET (NTC)     : {len(ntc):,}')
print('\nguides/gene (targeting):',
      targeting.target_gene_symbol.value_counts().value_counts().sort_index().to_dict())
targeting.head()

combined guides     : 78,370
targeting guides    : 77,412  across 19,529 genes
NO-TARGET (NTC)     : 958

guides/gene (targeting): {1: 85, 2: 124, 3: 201, 4: 19119}


,guide_id,protospacer,last19bp,target_gene_symbol,Set,guide_rank,gc_fraction,pool
0,A1BG-1,CAAGACAGGGAAGATGAAGC,AAGACAGGGAAGATGAAGC,A1BG,Set A,-1,0.50,pool1
1,A1BG-2,CACACCCCAGGCCACACCCC,ACACCCCAGGCCACACCCC,A1BG,Set A,-2,0.75,pool1
2,A1CF-1,CTGCATGGATGCAAGAGACA,TGCATGGATGCAAGAGACA,A1CF,Set A,-1,0.50,pool1
3,A1CF-2,TGACCCAATTATCTGGTCAA,GACCCAATTATCTGGTCAA,A1CF,Set A,-2,0.40,pool1
4,A2M-1,AAGATCTCTAAACAAAGTTG,AGATCTCTAAACAAAGTTG,A2M,Set A,-1,0.30,pool1


## Phase 1.5 — Reconcile outdated gene symbols to current Ensembl IDs

The library carries only legacy Calabrese/hCRISPRa-v2 gene **symbols** (no Ensembl IDs), and ~7% are
outdated (`ADRBK1`, `BAI2`, `B3GALTL`, `AARS`, …). We must fix these **before** twin detection: two
guides can target the same gene yet carry *different* labels (one old, one current), which would make a
real same-gene frameshift twin look like a cross-gene pair.

`reconcile_symbols_to_geneid` maps each symbol to a current `gene_id` + approved symbol using GENCODE v48
(direct) then HGNC previous/alias symbols (renames). Ambiguous (e.g. PAR X/Y genes with two IDs) and
unresolved symbols get `gene_id = NaN`, keep their old label, and are deferred to Phase 3 genomic
alignment — which is the authoritative locus-based check.

In [4]:
# Load reference tables (GENCODE v48 genes + HGNC complete set) and reconcile
gencode = pd.read_parquet(f'{GENOME}/gene_annotations_hg38.parquet')
hgnc = pd.read_csv(f'{GENOME}/hgnc_complete_set.txt', sep='\t', dtype=str, low_memory=False)

recon = reconcile_symbols_to_geneid(
    targeting['target_gene_symbol'], gencode_gene_df=gencode, hgnc_df=hgnc)

n_res = int(recon['gene_id'].notna().sum())
print(f'unique symbols          : {len(recon):,}')
print(f'resolved to Ensembl id  : {n_res:,} ({n_res / len(recon):.1%})')
print('\nresolution routes:')
print(recon['resolution'].value_counts().to_string())

# Attach gene_id + current_symbol to every targeting guide.
# gene_key = gene_id when resolved, else a per-symbol fallback so unresolved
# guides only group with their own old label (conservative, per the Phase-3-defer rule).
sym_map = recon.set_index('library_symbol')
targeting['gene_id'] = targeting['target_gene_symbol'].map(sym_map['gene_id'])
targeting['current_symbol'] = targeting['target_gene_symbol'].map(sym_map['current_symbol'])
targeting['name_resolution'] = targeting['target_gene_symbol'].map(sym_map['resolution'])
targeting['gene_key'] = targeting['gene_id'].where(
    targeting['gene_id'].notna(), 'SYM:' + targeting['target_gene_symbol'])

n_renamed = int((targeting['target_gene_symbol'] != targeting['current_symbol']).sum())
print(f'\nguides whose symbol changed: {n_renamed:,}')
targeting[targeting['target_gene_symbol'] != targeting['current_symbol']][
    ['guide_id', 'target_gene_symbol', 'current_symbol', 'gene_id', 'name_resolution']].head(8)

unique symbols          : 19,529
resolved to Ensembl id  : 19,321 (98.9%)

resolution routes:
resolution
v48_direct        18043
hgnc_prev          1241
unresolved          164
ambiguous_v48        40
hgnc_alias           25
hgnc_approved        12
ambiguous_prev        4

guides whose symbol changed: 4,678


,guide_id,target_gene_symbol,current_symbol,gene_id,name_resolution
28,AAED1-1,AAED1,PRXL2C,ENSG00000158122,hgnc_prev
29,AAED1-2,AAED1,PRXL2C,ENSG00000158122,hgnc_prev
44,AARS-1,AARS,AARS1,ENSG00000090861,hgnc_prev
45,AARS-2,AARS,AARS1,ENSG00000090861,hgnc_prev
306,ACN9-1,ACN9,SDHAF3,ENSG00000196636,hgnc_prev
307,ACN9-2,ACN9,SDHAF3,ENSG00000196636,hgnc_prev
350,ACPL2-1,ACPL2,PXYLP1,ENSG00000155893,hgnc_prev
351,ACPP-1,ACPP,ACP3,ENSG00000014257,hgnc_prev


## Phase 2 — Frameshift-twin detection (keyed on reconciled `gene_id`)

`frameshift_twin_pairs` indexes each guide by both of its 19 bp windows (`protospacer[:19]` and
`protospacer[1:]`) and returns pairs that share a 19-mer. We key "same gene" on the reconciled
**`gene_key`** (current `gene_id`, or the old symbol when unresolved) rather than the raw label, so
renamed same-gene pairs (`ADRBK1`/`GRK2`, `BAI2`/`ADGRB2`, …) are correctly counted as twins. `shift+1`
pairs are the redundant "1 bp apart" guides. (A 19 bp exact match is essentially impossible by chance —
4^19 ≈ 2.7e11 — so a shared window means the same genomic site; Phase 3 alignment confirms the ±1 bp
offset and settles any residual paralog ambiguity.)

In [5]:
# Same-gene twin pairs, keyed on reconciled gene_key (merges renamed labels)
twin_pairs = frameshift_twin_pairs(
    targeting, id_col='guide_id', seq_col='protospacer',
    gene_col='gene_key', same_gene_only=True)

# Add readable symbols alongside the gene_key used for matching
g2sym = targeting.set_index('guide_id')['target_gene_symbol']
twin_pairs = twin_pairs.rename(columns={'gene_a': 'gene_key_a', 'gene_b': 'gene_key_b'})
twin_pairs['symbol_a'] = twin_pairs['guide_a'].map(g2sym)
twin_pairs['symbol_b'] = twin_pairs['guide_b'].map(g2sym)
twin_pairs['renamed_pair'] = twin_pairs['symbol_a'] != twin_pairs['symbol_b']

n_involved = pd.unique(twin_pairs[['guide_a', 'guide_b']].values.ravel()).size
print(f'same-gene twin pairs : {len(twin_pairs):,}')
print(f'guides involved      : {n_involved:,} ({n_involved / len(targeting):.1%} of targeting)')
print('relation types       :', twin_pairs['relation'].value_counts().to_dict())
print(f'pairs recovered by name-fix (old vs new label): {int(twin_pairs["renamed_pair"].sum()):,}')
twin_pairs[twin_pairs['renamed_pair']].head(6)

same-gene twin pairs : 6,756
guides involved      : 12,832 (16.6% of targeting)
relation types       : {'shift+1': 6756}
pairs recovered by name-fix (old vs new label): 37


,guide_a,guide_b,gene_key_a,gene_key_b,same_gene,shared_19mer,relation,symbol_a,symbol_b,renamed_pair
73,ADRBK1-2,GRK2-4,ENSG00000173020,ENSG00000173020,True,AACGCCAGCGAGCCCGCGA,shift+1,ADRBK1,GRK2,True
223,B3GALTL-1,B3GLCT-2,ENSG00000187676,ENSG00000187676,True,GACGCTGGAAGCGCGCACA,shift+1,B3GALTL,B3GLCT,True
231,ADGRB2-1,BAI2-1,ENSG00000121753,ENSG00000121753,True,CGCGCCGCCTCCTGTTAAA,shift+1,ADGRB2,BAI2,True
256,BICRAL-1,GLTSCR1L-2,ENSG00000112624,ENSG00000112624,True,GCAGAGCATGAAGGCCGGG,shift+1,BICRAL,GLTSCR1L,True
257,BICRAL-2,GLTSCR1L-4,ENSG00000112624,ENSG00000112624,True,GGGCGCAGAGCATGAAGGC,shift+1,BICRAL,GLTSCR1L,True
280,BZRAP1-1,TSPOAP1-3,ENSG00000005379,ENSG00000005379,True,TGAACCGGCTGACAGGGTC,shift+1,BZRAP1,TSPOAP1,True


In [6]:
# Collapse twins into connected-component groups; keep the best-ranked guide per group.
# guide_rank is negative (closer to 0 = higher priority, e.g. -1 kept over -2).
rank = targeting.set_index('guide_id')['guide_rank']
twin_groups = collapse_twin_groups(twin_pairs, rank)

n_groups = twin_groups['twin_group_id'].nunique()
n_rep = int(twin_groups['is_representative'].sum())
n_collapsed = int((~twin_groups['is_representative']).sum())
print(f'twin groups          : {n_groups:,}')
print(f'representatives kept  : {n_rep:,}')
print(f'guides collapsed      : {n_collapsed:,}')
print('group size dist       :',
      twin_groups.groupby('twin_group_id').size().value_counts().sort_index().to_dict())
twin_groups.head()

twin groups          : 6,076
representatives kept  : 6,076
guides collapsed      : 6,756
group size dist       : {2: 5468, 3: 536, 4: 72}


,guide_id,twin_group_id,is_representative,collapsed_into
0,A2M-2,0,True,NaN
1,A2M-3,0,False,A2M-2
2,A3GALT2-2,1,True,NaN
3,A3GALT2-4,1,False,A3GALT2-2
4,A4GNT-1,2,True,NaN


### Cross-gene shared-window pairs (probe cross-hyb caveat — informational)

Guides for genes with **different reconciled `gene_id`** that still share a 19-mer window. After Phase 1.5,
renamed same-gene pairs have been removed from this set — what remains are genuine paralogs / adjacent
genes and `identical`-window multi-mapping guides. These are **not** collapsed (they are legitimately
different genes), but their capture probes can cross-hybridize, so a UMI assigned to one may partly
reflect the other. Saved for reference / downstream specificity QC.

In [7]:
# Cross-gene = shared 19-mer window but DIFFERENT reconciled gene_key.
# Because we key on gene_id, renamed same-gene pairs are no longer here; what
# remains are genuine paralogs / adjacent genes and multi-mapping (identical-window) guides.
all_shared = frameshift_twin_pairs(
    targeting, id_col='guide_id', seq_col='protospacer',
    gene_col='gene_key', same_gene_only=False)
crossgene_pairs = all_shared[~all_shared['same_gene']].reset_index(drop=True)
crossgene_pairs = crossgene_pairs.rename(columns={'gene_a': 'gene_key_a', 'gene_b': 'gene_key_b'})
crossgene_pairs['symbol_a'] = crossgene_pairs['guide_a'].map(g2sym)
crossgene_pairs['symbol_b'] = crossgene_pairs['guide_b'].map(g2sym)

print(f'cross-gene shared-window pairs: {len(crossgene_pairs):,}')
print('relation types               :', crossgene_pairs['relation'].value_counts().to_dict())
crossgene_pairs[['guide_a', 'guide_b', 'symbol_a', 'symbol_b', 'relation']].head(10)

cross-gene shared-window pairs: 76
relation types               : {'shift+1': 58, 'identical_head': 18}


,guide_a,guide_b,symbol_a,symbol_b,relation
0,ALPPL2-2,ALPP-3,ALPPL2,ALPP,identical_head
1,FAM47B-1,FAM47C-4,FAM47B,FAM47C,identical_head
2,IFNA10-1,IFNA16-3,IFNA10,IFNA16,identical_head
3,LCE2B-2,LCE2C-2,LCE2B,LCE2C,identical_head
4,MAGEA2-1,MAGEA3-1,MAGEA2,MAGEA3,identical_head
5,NBPF4-1,NBPF6-1,NBPF4,NBPF6,identical_head
6,OR10H2-2,OR10H5-1,OR10H2,OR10H5,identical_head
7,OR2A1-2,OR2A42-1,OR2A1,OR2A42,identical_head
8,OR2T34-2,OR2T3-3,OR2T34,OR2T3,identical_head
9,OR3A3-1,OR3A2-4,OR3A3,OR3A2,identical_head


### Annotate the master table and build the de-duplicated library

Add twin flags to every targeting guide, then build the de-duplicated library = non-twin guides +
one representative per twin group. Also report, per gene, how many independent guides survive dedup
(collapsing a size-4 twin group can drop a 4-guide gene to a single effective guide — worth watching).

In [8]:
# Merge twin/group flags onto the master targeting table (targeting already carries
# gene_id / current_symbol / gene_key / name_resolution from Phase 1.5)
master = targeting.merge(twin_groups, on='guide_id', how='left')
master['is_twin'] = master['twin_group_id'].notna()
# non-twin guides are trivially their own representative; only twins can be collapsed
master['is_representative'] = (~master['is_twin']) | (master['is_representative'] == True)
master['keep'] = master['is_representative']

# De-duplicated library = everything we keep
dedup = master[master['keep']].reset_index(drop=True)

print(f'master targeting guides : {len(master):,}')
print(f'  flagged as twin       : {int(master["is_twin"].sum()):,}')
print(f'  collapsed (dropped)   : {int((~master["keep"]).sum()):,}')
print(f'deduplicated library    : {len(dedup):,} guides across {dedup.gene_key.nunique():,} genes')

# Per-gene (keyed on reconciled gene_key so renamed old/new labels count as one gene)
per_gene = (master.groupby('gene_key')
            .agg(current_symbol=('current_symbol', 'first'),
                 designed_symbols=('target_gene_symbol', lambda s: '|'.join(sorted(set(s)))),
                 gene_id=('gene_id', 'first'),
                 n_designed=('guide_id', 'size'),
                 n_twin=('is_twin', 'sum'),
                 n_collapsed=('keep', lambda s: int((~s).sum())),
                 n_after_dedup=('keep', 'sum'))
            .reset_index()
            .sort_values('n_after_dedup'))
print('\nguides/gene AFTER dedup:',
      per_gene['n_after_dedup'].value_counts().sort_index().to_dict())
n_singleton = int((per_gene['n_after_dedup'] == 1).sum())
print(f'genes reduced to a single effective guide: {n_singleton:,}')
per_gene.head(10)

master targeting guides : 77,412
  flagged as twin       : 12,832
  collapsed (dropped)   : 6,756
deduplicated library    : 70,656 guides across 19,150 genes



guides/gene AFTER dedup: {1: 125, 2: 777, 3: 5026, 4: 12854, 5: 59, 6: 83, 7: 120, 8: 105, 10: 1}
genes reduced to a single effective guide: 125


,gene_key,current_symbol,designed_symbols,gene_id,n_designed,n_twin,n_collapsed,n_after_dedup
7908,ENSG00000141034,GID4,GID4,ENSG00000141034,1,0,0,1
15475,ENSG00000186827,TNFRSF4,TNFRSF4,ENSG00000186827,4,4,3,1
15765,ENSG00000188290,HES4,HES4,ENSG00000188290,4,4,3,1
5242,ENSG00000122862,SRGN,SRGN,ENSG00000122862,4,4,3,1
12613,ENSG00000170948,MBD3L1,MBD3L1,ENSG00000170948,4,4,3,1
15364,ENSG00000186260,MRTFB,MKL2,ENSG00000186260,4,4,3,1
16962,ENSG00000204291,COL15A1,COL15A1,ENSG00000204291,4,4,3,1
11774,ENSG00000167195,GOLGA6C,GOLGA6C,ENSG00000167195,1,0,0,1
18560,ENSG00000263513,FAM72C,FAM72C,ENSG00000263513,1,0,0,1
18754,ENSG00000275221,H2AC15,HIST1H2AK,ENSG00000275221,1,0,0,1


In [9]:
# Reconciliation side-effect: genes listed in the library under MULTIPLE legacy aliases.
# These collapse to one gene_id, so the same gene carried >4 designed guides (e.g. SEM1 =
# C7orf76 = SHFM1, 12 guides). Downstream per-gene analysis should treat each as one target.
multi_alias = per_gene[per_gene['designed_symbols'].str.contains(r'\|')].copy()
print(f'genes listed under >1 legacy alias : {len(multi_alias):,}')
print(f'  guides involved                  : {int(multi_alias["n_designed"].sum()):,}')
print('  alias-count distribution         :',
      multi_alias['designed_symbols'].str.count(r'\|').add(1).value_counts().sort_index().to_dict())
multi_alias.sort_values('n_designed', ascending=False)[
    ['current_symbol', 'designed_symbols', 'gene_id', 'n_designed', 'n_after_dedup']].head(8)

genes listed under >1 legacy alias : 378
  guides involved                  : 2,704
  alias-count distribution         : {2: 377, 3: 1}


,current_symbol,designed_symbols,gene_id,n_designed,n_after_dedup
5857,SEM1,C7orf76|SEM1|SHFM1,ENSG00000127922,12,10
9672,NIFK,MKI67IP|NIFK,ENSG00000155438,8,8
9833,ADGRG4,ADGRG4|GPR112,ENSG00000156920,8,6
11153,TEX47,C7orf62|TEX47,ENSG00000164645,8,7
10159,DRC7,CCDC135|DRC7,ENSG00000159625,8,7
5182,LDB3,LDB3|ZASP,ENSG00000122367,8,7
11238,WASHC5,KIAA0196|WASHC5,ENSG00000164961,8,7
13727,TYMSOS,C18orf56|TYMSOS,ENSG00000176912,8,7


### Save outputs

`master` (all targeting guides + twin flags) is the input to Phase 3 alignment. `dedup` is the
collapsed library. Twin pairs/groups and the cross-gene and per-gene tables are saved for review.

In [10]:
def save(df, stem):
    df.to_parquet(f'{RESULTS}/{stem}.parquet')
    df.to_csv(f'{RESULTS}/{stem}.csv', index=False)
    print(f'  {stem:<44} {len(df):>7,} rows')

print('Writing to results/ ...')
save(recon,           'CRISPRa_symbol_to_geneid_reconciliation')  # old symbol -> current id/symbol
save(master,          'CRISPRa_targeting_sgRNA_master')       # all guides + ids + twin flags (Phase 3 input)
save(dedup,           'CRISPRa_targeting_sgRNA_deduplicated')  # collapsed library
save(ntc,             'CRISPRa_NO-TARGET_sgRNA')               # NTC block (annotated later)
save(twin_pairs,      'CRISPRa_frameshift_twin_pairs')
save(twin_groups,     'CRISPRa_frameshift_twin_groups')
save(crossgene_pairs, 'CRISPRa_crossgene_sharedwindow_pairs')
save(per_gene,        'CRISPRa_per_gene_dedup_summary')
save(multi_alias,     'CRISPRa_genes_multiple_legacy_aliases')  # same gene listed under >1 old symbol
print('Done.')

Writing to results/ ...
  CRISPRa_symbol_to_geneid_reconciliation       19,529 rows


  CRISPRa_targeting_sgRNA_master                77,412 rows


  CRISPRa_targeting_sgRNA_deduplicated          70,656 rows
  CRISPRa_NO-TARGET_sgRNA                          958 rows
  CRISPRa_frameshift_twin_pairs                  6,756 rows
  CRISPRa_frameshift_twin_groups                12,832 rows
  CRISPRa_crossgene_sharedwindow_pairs              76 rows
  CRISPRa_per_gene_dedup_summary                19,150 rows
  CRISPRa_genes_multiple_legacy_aliases            378 rows
Done.
